In [ ]:
#Import all necessary libraries
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from hashlib import sha256
import json, gc, zipfile
import numpy as np
import tensorflow as tf
from sklearn.metrics import (classification_report, confusion_matrix,
    accuracy_score, f1_score, balanced_accuracy_score, roc_auc_score,
    average_precision_score)

In [ ]:
MAPPING = {**dict.fromkeys(['NV','BKL','DF','VASC'], 'benign'),
           **dict.fromkeys(['MEL','BCC','SCC'], 'malignant')}

fingerprint_path = Path.cwd().parent / "artifacts" / "dataset_fingerprint.json"

with open(fingerprint_path) as f:
    fingerprint_data = json.load(f)
fingerprint = fingerprint_data["fingerprint"]

In [ ]:
data_path = Path.cwd().parent / "artifacts" / "isic2019_binary_split.csv"
out = Path.cwd().parent /'artifacts'
out.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(data_path)
print(df.head())
print(df.columns)
print(df["split"].value_counts())
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))

In [ ]:
#Prepare training & validation dataframes
train = df[df["split"] == "train"].copy()
val   = df[df["split"] == "val"].copy()
test  = df[df["split"] == "test"].copy()

assert len(train) > 0
assert len(val) > 0
assert len(test) > 0

In [ ]:
# Calculate class weights
class_weights = {
    int(k): len(train) / (2 * int(v))
    for k, v in train.target.value_counts().items()
}

print(pd.crosstab(df.label, df.split, margins=True))
print('Class weights:', class_weights)

In [ ]:
#Check available hardware (GPU vs CPU)
#DISCUSS ON THURSDAY
#assert tf.config.list_physical_devices('GPU'), 'Enable GPU before training.'
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print("GPU detected:", gpus)
else:
    print("No compatible GPU detected. Running on CPU.")

print("TensorFlow devices:", tf.config.list_physical_devices())

In [ ]:
#Define image loading
def load_image(path, target):
    x = tf.io.decode_image(tf.io.read_file(path), channels=3,
                           expand_animations=False)
    x.set_shape([None, None, 3])
    return tf.cast(tf.image.resize(x, [128,128]), tf.float32)/255., target

In [ ]:
#Define TensorFlow dataset function
def dataset(part, seed, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((part.path.astype(str).to_numpy(),
                                            part.target.to_numpy('int32')))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(len(part), seed=seed, reshuffle_each_iteration=True)
    opts = tf.data.Options()
    opts.experimental_deterministic = True
    return ds.with_options(opts).batch(32).prefetch(tf.data.AUTOTUNE)

In [ ]:
#Separate train & val datasets for grid search
train_ds = dataset(train, seed=42, shuffle=True)
val_ds = dataset(val, seed=42, shuffle=False)

In [ ]:
#Grid search
def hyperparameter_search_cnn(filters, kernel_sizes,
                              dense_units, dropout_rates,
                              learning_rates):

    val_acc_list = []

    for i in filters:
        for j in kernel_sizes:
            for k in dense_units:
                for z in dropout_rates:
                    for lr in learning_rates:

                        tf.keras.backend.clear_session()
                        tf.keras.utils.set_random_seed(42)

                        model = build_cnn(
                            filters=i,
                            kernel_size=j,
                            dense_units=k,
                            dropout_rate=z
                        )

                        model.compile(
                            optimizer=tf.keras.optimizers.Adam(
                                learning_rate=lr
                            ),
                            loss='binary_crossentropy',
                            metrics=['accuracy']
                        )

                        history = model.fit(
                            train_ds,
                            validation_data=val_ds,
                            epochs=25,
                            class_weight=class_weights,
                            callbacks=[
                                tf.keras.callbacks.EarlyStopping(
                                    monitor='val_loss',
                                    patience=5,
                                    restore_best_weights=True
                                )
                            ],
                            verbose=0
                        )

                        y_val_pred = (
                            model.predict(val_ds, verbose=0).reshape(-1) >= 0.5
                        ).astype(int)

                        y_val = val.target.to_numpy()

                        output = {
                            "filters": str(i),
                            "kernel_size": j,
                            "dense_units": k,
                            "dropout_rate": z,
                            "learning_rate": lr,
                            "val_acc": accuracy_score(y_val, y_val_pred),
                            "val_f1": f1_score(
                                y_val, y_val_pred,
                                average='macro',
                                zero_division=0
                            ),
                            "val_auc": roc_auc_score(
                                y_val,
                                model.predict(val_ds, verbose=0).reshape(-1)
                            ),
                            "val_loss": min(history.history['val_loss'])
                        }

                        val_acc_list.append(output)

    df_output = pd.DataFrame(val_acc_list)
    df_output["Model"] = range(1, df_output.shape[0] + 1)

    return df_output

In [ ]:
filters = [
    (32, 64, 128, 256),
    (32, 64, 128, 128)
]

kernel_sizes = [3, 5]

dense_units = [64, 128]

dropout_rates = [0.25, 0.35, 0.45]

learning_rates = [0.001, 0.0001]

In [ ]:
df_cnn = hyperparameter_search_cnn(
    filters,
    kernel_sizes,
    dense_units,
    dropout_rates,
    learning_rates
)

print(df_cnn.sort_values("val_acc", ascending=False))

In [ ]:
best_model = df_cnn.loc[df_cnn["val_loss"].idxmin()]
print(best_model)

In [ ]:
#Build a CNN model 
def build_cnn():
    return tf.keras.Sequential([
        tf.keras.Input((128,128,3)),

        tf.keras.layers.Conv2D(32,3,padding='same',activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64,3,padding='same',activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128,3,padding='same',activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(256,3,padding='same',activation='relu'),
        
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(128,activation='relu'),
        tf.keras.layers.Dropout(.35,seed=12345),
        
        tf.keras.layers.Dense(1,activation='sigmoid')])

In [ ]:
#Run the model 
results = []
for seed in [42,43,44]:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    model = build_cnn()
    init_hash = sha256(b''.join(w.tobytes() for w in model.get_weights())).hexdigest()
    # Preserve initial arrays for future paired binary augmentation comparisons.
    np.savez_compressed(out/f'initial_weights_seed{seed}.npz', *model.get_weights())
    model.compile(optimizer=tf.keras.optimizers.Adam(.001),
                  loss='binary_crossentropy', metrics=['accuracy'])
    run = f'binary_no_aug_seed{seed}'
    checkpoint = out/f'{run}.keras'
    train_ds, val_ds = dataset(train, seed, True), dataset(val, seed, False)
    history = model.fit(train_ds, validation_data=val_ds, epochs=25,
        class_weight=class_weights, callbacks=[
            tf.keras.callbacks.ModelCheckpoint(str(checkpoint),
                monitor='val_loss',save_best_only=True),
            tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=5,
                                             restore_best_weights=True)], verbose=2)
    best = tf.keras.models.load_model(checkpoint)
    probability = best.predict(val_ds,verbose=0).reshape(-1)
    y = val.target.to_numpy()
    pred = (probability >= .5).astype(int)
    tn,fp,fn,tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
    row = dict(seed=seed,condition='binary_no_aug',
        epochs_trained=len(history.history['loss']),
        selected_epoch=int(np.argmin(history.history['val_loss'])+1),
        validation_loss=float(min(history.history['val_loss'])),
        accuracy=float(accuracy_score(y,pred)),
        macro_f1=float(f1_score(y,pred,labels=[0,1],average='macro',zero_division=0)),
        balanced_accuracy=float(balanced_accuracy_score(y,pred)),
        malignant_sensitivity=float(tp/(tp+fn)), specificity=float(tn/(tn+fp)),
        roc_auc=float(roc_auc_score(y,probability)),
        average_precision=float(average_precision_score(y,probability)),
        threshold=.5,initial_weights_sha256=init_hash,
        split_fingerprint=fingerprint,test_evaluated=False)
    report=classification_report(y,pred,labels=[0,1],
        target_names=['benign','malignant'],output_dict=True,zero_division=0)
    (out/f'{run}_metrics.json').write_text(json.dumps(
        dict(**row,classification_report=report),indent=2))
    pd.DataFrame(history.history).assign(
        epoch=lambda d: np.arange(1,len(d)+1)).to_csv(out/f'{run}_history.csv',index=False)
    predictions=val[['image_id','original_label','label','group']].copy()
    predictions['malignant_probability']=probability
    predictions['prediction']=pred
    predictions.to_csv(out/f'{run}_validation_predictions.csv',index=False)
    pd.DataFrame(confusion_matrix(y,pred,labels=[0,1]),
        index=['benign','malignant'],columns=['benign','malignant']).to_csv(
            out/f'{run}_confusion_matrix.csv')
    results.append(row)
    pd.DataFrame(results).to_csv(out/'binary_baseline_results.csv',index=False)
    # Checkpoint each seed's complete evidence before moving to the next run.
    
    # Checkpoint each seed's complete evidence before moving to the next run.
    archive_path = out / f'{run}_artifacts.zip'

    with zipfile.ZipFile(archive_path, 'w',
                     zipfile.ZIP_DEFLATED) as archive:
        for p in out.glob(f'{run}*'):
            archive.write(p, p.name)

        for name in [f'initial_weights_seed{seed}.npz',
                 'isic2019_binary_split.csv']:
            archive.write(out / name, name)

print("Archive saved:", archive_path)
print('BINARY_RUN_COMPLETE',json.dumps(row),flush=True)
del model,best,train_ds,val_ds
gc.collect()

In [ ]:
#Save everything
config=dict(mapping=MAPPING,class_weights=class_weights,seeds=[42,43,44],
    tensorflow_version=tf.__version__,split_fingerprint=fingerprint,
    threshold=.5,test_evaluated=False,max_epochs=25,patience=5,
    batch_size=32,steps_per_epoch=31,learning_rate=.001)
(out/'binary_baseline_config.json').write_text(json.dumps(config,indent=2))
print('ALL_BINARY_BASELINES_COMPLETE')
print(pd.DataFrame(results).to_string(index=False))
# No test image dataset is constructed or evaluated in this script.
